# Project 06 -- Copula Dependency Modeling

Copula-based dependence structures for multi-asset risk aggregation and scenario generation.

**Outline:**
1. Generate correlated data, show scatter of raw returns vs PIT uniforms
2. Fit all 5 copula families, show comparison table (tail dependence)
3. Scatter plots: empirical vs simulated for Gaussian, t, Clayton
4. Portfolio VaR under each copula assumption -- bar chart comparison
5. Key insight: Gaussian copula underestimates tail risk

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from scipy import stats

# Ensure project src/ is importable
_PROJECT_SRC = str(Path.cwd().parent / "src")
if _PROJECT_SRC not in sys.path:
    sys.path.insert(0, _PROJECT_SRC)

# Shared library
from risk_analyst.models.copula import (
    gaussian_copula_fit,
    t_copula_fit,
    clayton_copula_fit,
    gumbel_copula_fit,
    frank_copula_fit,
    copula_sample,
    tail_dependence,
    pit_transform,
)

# Project-local
from model import CopulaModel
from diagnostics import (
    plot_copula_scatter,
    plot_simulated_vs_empirical,
    plot_tail_dependence_comparison,
    plot_var_by_copula,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120

print("Imports OK.")

In [ ]:
# Load config
config_path = Path.cwd().parent / "configs" / "default.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Config loaded:")
config

## 1. Generate Correlated Data and PIT Uniforms

We fetch multi-asset returns, then show the difference between raw return scatter plots and the pseudo-uniform marginals obtained by the probability integral transform (PIT).

In [ ]:
# Fetch prices and compute returns
from risk_analyst.data.market import fetch_prices, compute_returns

tickers = config["data"]["tickers"]
prices = fetch_prices(
    tickers=tickers,
    start_date=config["data"]["start_date"],
    end_date=config["data"]["end_date"],
)
returns = compute_returns(prices, method=config["data"]["returns_type"])

print(f"Assets: {list(returns.columns)}")
print(f"Observations: {len(returns)}, from {returns.index[0].date()} to {returns.index[-1].date()}")
returns.describe().round(5)

In [ ]:
# Apply PIT to raw returns (empirical, no GARCH filtering for illustration)
u_raw = pit_transform(returns.values, method="empirical")

# Scatter: raw returns vs PIT uniforms for SPY-QQQ pair
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(returns.iloc[:, 0], returns.iloc[:, 1], s=3, alpha=0.3, color="steelblue")
axes[0].set_xlabel(f"{tickers[0]} return")
axes[0].set_ylabel(f"{tickers[1]} return")
axes[0].set_title("Raw Returns")

axes[1].scatter(u_raw[:, 0], u_raw[:, 1], s=3, alpha=0.3, color="coral")
axes[1].set_xlabel(f"{tickers[0]} (PIT)")
axes[1].set_ylabel(f"{tickers[1]} (PIT)")
axes[1].set_title("Pseudo-Uniform (PIT)")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

fig.suptitle(f"{tickers[0]} vs {tickers[1]}: Raw Returns and PIT Uniforms", fontsize=13)
fig.tight_layout()
plt.show()

print("Note: the PIT scatter reveals the copula structure, stripped of marginal effects.")

## 2. Fit All Copula Families and Compare Tail Dependence

We fit five copula families (Gaussian, Student-t, Clayton, Gumbel, Frank) to the pseudo-uniform data and compare their tail dependence properties.

**Tail dependence** measures the probability that both variables take extreme values simultaneously:
- $\lambda_L = \lim_{u \to 0^+} P(U_2 \leq u \mid U_1 \leq u)$ (lower tail)
- $\lambda_U = \lim_{u \to 1^-} P(U_2 > u \mid U_1 > u)$ (upper tail)

In [ ]:
# Instantiate CopulaModel
cm = CopulaModel(config)

# Use bivariate PIT data for demonstration (SPY vs QQQ)
u_biv = u_raw[:, :2]

# Fit all families
fitted = cm.fit_all_families(u_biv)
print("Fitted copula families:", list(fitted.keys()))
print()

# Show fitted parameters
for family, params in fitted.items():
    print(f"  {family}:")
    for k, v in params.items():
        if k == "corr_matrix":
            print(f"    rho = {v[0, 1]:.4f}")
        elif k not in ("family", "d"):
            print(f"    {k} = {v:.4f}" if isinstance(v, float) else f"    {k} = {v}")

In [ ]:
# Comparison table: tail dependence coefficients
comparison_df = cm.compare_families(u_biv)
print("Copula Family Comparison:")
comparison_df

In [ ]:
# Bar chart of tail dependence
fig, ax = plot_tail_dependence_comparison(comparison_df)
plt.show()

print("Key observation: Only the t-copula captures BOTH lower and upper tail dependence.")
print("Clayton captures lower tail only; Gumbel captures upper tail only.")
print("Gaussian and Frank have ZERO tail dependence -- they miss extreme co-movements.")

## 3. Empirical vs Simulated Scatter Plots

For each fitted copula, we simulate 5,000 samples and compare the scatter structure with the empirical data. A well-fitted copula should produce a scatter pattern visually similar to the data.

In [ ]:
# Compare empirical vs simulated for Gaussian, t, Clayton
for family in ["gaussian", "t", "clayton"]:
    u_sim = cm.simulate_joint(family, n_samples=5000)
    fig, axes = plot_simulated_vs_empirical(u_biv, u_sim, family)
    plt.show()

In [ ]:
# Also show the full copula scatter for each family
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

# Empirical
axes[0].scatter(u_biv[:, 0], u_biv[:, 1], s=2, alpha=0.3, color="steelblue")
axes[0].set_title("Empirical")
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

# Simulated from each family
for i, family in enumerate(["gaussian", "t", "clayton", "gumbel", "frank"]):
    u_sim = cm.simulate_joint(family, n_samples=5000)
    colors = ["coral", "purple", "forestgreen", "goldenrod", "teal"]
    axes[i + 1].scatter(u_sim[:, 0], u_sim[:, 1], s=2, alpha=0.3, color=colors[i])
    axes[i + 1].set_title(f"{family.capitalize()}")
    axes[i + 1].set_xlim(0, 1); axes[i + 1].set_ylim(0, 1)

fig.suptitle("Empirical vs Simulated Copulas (SPY-QQQ)", fontsize=14)
fig.tight_layout()
plt.show()

## 4. Portfolio VaR under Each Copula Assumption

We now compute portfolio VaR under each copula family using the full pipeline:
1. Fit GARCH(1,1) to each asset's returns (marginal filtering)
2. Apply PIT to standardised residuals
3. Fit copula to the pseudo-uniform data
4. Simulate 10,000 joint scenarios from the copula
5. Invert PIT + GARCH to get return scenarios
6. Compute portfolio VaR

This shows how the choice of copula affects the estimated tail risk.

In [ ]:
# Full pipeline: GARCH marginals -> copula -> VaR
weights = np.array(config["portfolio"]["weights"])
alphas = config["portfolio"]["confidence_levels"]

cm_full = CopulaModel(config)
var_df = cm_full.portfolio_var_by_copula(returns, weights, alphas)

print("Portfolio VaR by Copula Family:")
print(f"  Weights: {weights}")
print(f"  Confidence levels: {alphas}")
print()
var_df

In [ ]:
# Bar chart: VaR by copula and alpha
fig, ax = plot_var_by_copula(var_df)
plt.show()

In [ ]:
# Compare Gaussian vs t-copula VaR at 99%
var_99_col = "VaR_0.99"
gaussian_var = var_df.loc[var_df["family"] == "gaussian", var_99_col].values[0]
t_var = var_df.loc[var_df["family"] == "t", var_99_col].values[0]

print(f"99% VaR Comparison:")
print(f"  Gaussian copula: {gaussian_var:.6f}")
print(f"  t-copula:        {t_var:.6f}")
if t_var > gaussian_var:
    pct_diff = (t_var / gaussian_var - 1) * 100
    print(f"  t-copula VaR is {pct_diff:.1f}% higher than Gaussian")
    print()
    print("  --> The Gaussian copula UNDERESTIMATES tail risk because it assumes")
    print("      zero tail dependence (extreme co-movements don't cluster).")

## 5. Key Insight: Gaussian Copula Underestimates Tail Risk

The Gaussian copula was infamously at the center of the 2008 financial crisis (see *"The Formula That Killed Wall Street"*, Wired, 2009). Its core flaw:

**Zero tail dependence.** The Gaussian copula assumes that the probability of simultaneous extreme losses approaches zero. In reality, financial assets exhibit *increased* correlation during crises -- precisely when diversification is needed most.

| Property | Gaussian | Student-t | Clayton | Gumbel | Frank |
|----------|----------|-----------|---------|--------|-------|
| $\lambda_L$ (lower) | 0 | > 0 | > 0 | 0 | 0 |
| $\lambda_U$ (upper) | 0 | > 0 | 0 | > 0 | 0 |
| Crisis relevance | Misses joint crashes | Captures joint extremes | Lower tail only | Upper tail only | Misses both tails |

**Practical implications:**
- Using a Gaussian copula for risk management **systematically understates** portfolio VaR and ES at high confidence levels (99%, 99.5%).
- The t-copula with low degrees of freedom provides a more conservative and realistic model for stress scenarios.
- For portfolios with downside protection needs, the Clayton copula captures the asymmetric lower-tail clustering observed in equity markets.
- A robust risk framework should **compare VaR across copula assumptions** rather than relying on a single model.

In [ ]:
# Final summary: full tail dependence + VaR table
summary = comparison_df.merge(var_df, on="family", how="left")
print("Complete Copula Analysis Summary:")
summary